# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniyalhaider236/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



I fit the same Logistic Regression architecture validated in Weeks 5–6 on the **full 30,000-row dataset**
(not just a train split) — the goal here isn't to re-measure accuracy, it's to score every page for a
usable playbook. Its measured reliability is what Week 6 already found: Precision@20 averaging 0.68 across
five different client holdouts, ranging 0.50–0.75. That range, not a single number, is what I'm relying on.

**Two dimensions build five archetypes:**
- **Risk tier** — the model's `decline_proba`, split into thirds (low / medium / high) across the dataset
- **Visibility** — `impressions_90d >= 300`, the same "visible" cutoff from Week 4

| Archetype | Risk | Visible | Reason code | Action |
|---|---|---|---|---|
| Priority Refresh | High | Yes | `high_risk_and_visible` | `review_for_refresh` |
| Quiet Decline | High | No | `high_risk_low_visibility` | `monitor_low_priority` |
| Watch | Medium | Either | `medium_risk` | `watch_next_cycle` |
| Protect | Low | Yes | `low_risk_and_visible` | `protect_periodic_check` |
| Low Priority | Low | No | `low_risk_low_visibility` | `monitor_only` |

Within each archetype, pages are ranked by `expected_impact_score = decline_proba × impressions_90d` — the
Week-4 lesson (volume is an impact multiplier, not a decline predictor on its own) carried forward, now on
top of a validated risk score instead of a raw threshold rule. **The queue works top-to-bottom by
archetype tier first** (finish Priority Refresh before ever looking at Protect), not by a single blended
score — a page with a huge audience but low risk shouldn't jump ahead of an actually-at-risk page just
because it's bigger.

Sanity check: decline rate falls monotonically from Priority Refresh (66.4%) through Quiet Decline (58.4%)
and Watch (59.7%) down to Protect (45.0%) and Low Priority (31.4%) — the tiers separate real outcomes in
the expected direction, even though (as Week 6 showed) no single split's exact numbers should be over-trusted.

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

honest_num = ["content_age_days", "days_since_last_update", "avg_position", "ctr",
              "word_count", "search_volume", "competition"]
for c in ["word_count", "search_volume", "competition"]:
    df[c + "_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())
honest_features = honest_num + [c + "_missing" for c in ["word_count", "search_volume", "competition"]]
ct_dummies = pd.get_dummies(df["content_type"], prefix="ctype")
X = pd.concat([df[honest_features], ct_dummies], axis=1)
y = df["is_declining_label"]

# Final playbook model: same validated architecture, fit on the full dataset
scaler = StandardScaler()
X_s = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
lr_final = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_final.fit(X_s, y)
df["decline_proba"] = lr_final.predict_proba(X_s)[:, 1]

q1, q2 = df["decline_proba"].quantile([1/3, 2/3])
df["risk_tier"] = df["decline_proba"].apply(lambda p: "high" if p >= q2 else ("medium" if p >= q1 else "low"))
df["visible"] = df["impressions_90d"] >= 300

def archetype(row):
    if row["risk_tier"] == "high" and row["visible"]: return "Priority Refresh"
    if row["risk_tier"] == "high" and not row["visible"]: return "Quiet Decline"
    if row["risk_tier"] == "medium": return "Watch"
    if row["risk_tier"] == "low" and row["visible"]: return "Protect"
    return "Low Priority"

df["archetype"] = df.apply(archetype, axis=1)
reason_map = {"Priority Refresh": "high_risk_and_visible", "Quiet Decline": "high_risk_low_visibility",
              "Watch": "medium_risk", "Protect": "low_risk_and_visible", "Low Priority": "low_risk_low_visibility"}
action_map = {"Priority Refresh": "review_for_refresh", "Quiet Decline": "monitor_low_priority",
              "Watch": "watch_next_cycle", "Protect": "protect_periodic_check", "Low Priority": "monitor_only"}
df["reason_code"] = df["archetype"].map(reason_map)
df["action_label"] = df["archetype"].map(action_map)
df["expected_impact_score"] = df["decline_proba"] * df["impressions_90d"]

archetype_priority = {"Priority Refresh": 0, "Quiet Decline": 1, "Watch": 2, "Protect": 3, "Low Priority": 4}
df["_priority"] = df["archetype"].map(archetype_priority)
df = df.sort_values(["_priority", "expected_impact_score"], ascending=[True, False]).reset_index(drop=True)
df["rank"] = df.index + 1
df = df.drop(columns=["_priority"])

print(df["archetype"].value_counts().reindex(["Priority Refresh","Quiet Decline","Watch","Protect","Low Priority"]))
print()
print(df.groupby("archetype", sort=False)["is_declining_label"].agg(n="count", decline_rate="mean")
      .reindex(["Priority Refresh","Quiet Decline","Watch","Protect","Low Priority"]))
print()
print(df[["content_id","client_id","rank","archetype","reason_code","action_label","decline_proba"]].head(10).to_string(index=False))

archetype
Priority Refresh     7162
Quiet Decline        2838
Watch               10000
Protect              5409
Low Priority         4591
Name: count, dtype: int64

                      n  decline_rate
archetype                            
Priority Refresh   7162      0.664200
Quiet Decline      2838      0.583862
Watch             10000      0.597100
Protect            5409      0.449991
Low Priority       4591      0.314311

          content_id         client_id  rank        archetype           reason_code       action_label  decline_proba
content_2cb567c3c89b client_6208ef0f77     1 Priority Refresh high_risk_and_visible review_for_refresh       0.600930
content_2dba2b1f9536 client_6208ef0f77     2 Priority Refresh high_risk_and_visible review_for_refresh       0.578729
content_cb112fce36be client_19581e27de     3 Priority Refresh high_risk_and_visible review_for_refresh       0.670605
content_44e481c8f55b client_19581e27de     4 Priority Refresh high_risk_and_visible review_for

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



**Who this is for:** a content team prioritizing *review order* across their own existing page portfolio —
"what should someone look at first," never "what should change automatically." This is a triage tool, not
a publishing decision.

**What it does NOT claim:** it does not measure or prove that refreshing a page *causes* better
performance — that's exactly the methodology question I raised about the paper's own Freshness Multiplier
finding in Week 6, and I'm holding this playbook to the same standard I asked of that one. This ranks
*risk*, not refresh ROI.

**Where it stops being valid:**
- Only tested on pages resembling this dataset — established content with real history. New pages
  (`content_age_days` near 0) and pages missing `word_count`/`search_volume`/`competition` are working off
  imputed values and a missingness flag, not real measurements — treat scores for the ~26% of pages
  missing `word_count` as lower-confidence.
- Reliability is a *range*, not a point: Precision@20 measured 0.50–0.75 across different client holdouts
  in Week 6. Any single re-run's exact number should be read against that range, not treated as new truth.
- Built on a snapshot with a specific label threshold (this file's `trend_direction` uses roughly a ±20%
  cutoff, not the ±10% the FlyRank paper discloses — see Week 6). If a future data export changes that
  threshold, decline rates and every downstream number shift with it.

In [8]:
print("Rows with imputed/missing supporting data (lower-confidence scores):")
for c in ["word_count", "search_volume", "competition"]:
    n_missing = df[c + "_missing"].sum()
    print(f"  {c}: {n_missing} rows ({n_missing/len(df):.1%})")

print(f"\nContent age distribution (checking for 'too new to trust' pages):")
print(df["content_age_days"].describe()[["min","25%","50%","max"]])
print(f"Pages under 30 days old: {(df['content_age_days'] < 30).sum()}")


Rows with imputed/missing supporting data (lower-confidence scores):
  word_count: 7699 rows (25.7%)
  search_volume: 2468 rows (8.2%)
  competition: 2468 rows (8.2%)

Content age distribution (checking for 'too new to trust' pages):
min     90.0
25%    132.0
50%    236.0
max    564.0
Name: content_age_days, dtype: float64
Pages under 30 days old: 0


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



**Before acting on any "Priority Refresh" or "Quiet Decline" page, a human must check:**
- Real content quality and relevance — especially the ~26% of pages missing `word_count`, where I can't
  confirm from data alone whether it's a substantial article or thin/aggregated content
- Technical or indexing problems first — Week 4 found pages with implausibly low CTR at good positions
  (down to 0.00% at ~200k impressions); that's a tracking or snippet issue a content rewrite won't fix
- Whether the underlying keyword's search demand is itself fading — no refresh fixes a dying topic
- Whether the page is already scheduled for intentional retirement or merging into another page

**What should NEVER be automated from this playbook:**
- No auto-publishing or auto-editing of content — every action here ends with a human decision
- No treating Precision@20 ≈ 0.68–0.75 as a guarantee for any *specific* page — it's a cohort-level
  measurement, not an individual one
- No using this to evaluate or penalize a specific writer, editor, or team
- No acting on a small client-level slice (n < 200) without a manual sanity check — Week 4's smallest
  staleness buckets held under 200 rows each and were explicitly unstable

**A concrete reason for that last rule, found while building this:** I checked how many pages share a
near-identical profile (same content age, same days-since-update, same CTR, rounded) with at least 5 other
pages that ended up with *different* real outcomes. **919 such profiles cover 16,858 rows — 56% of the
entire dataset.** That means for more than half the queue, the model can rank the *cohort* reliably (the
archetype decline rates above are cleanly separated) but often cannot tell which specific pages *within*
an identical-looking cohort are the real problem. Treat archetype membership as the trustworthy signal;
treat exact rank position within a tied cohort as much weaker.

In [9]:
profile_cols = ["content_age_days", "days_since_last_update", "ctr"]
dup = df.groupby(profile_cols)["is_declining_label"].agg(["count", "nunique", "mean"])
mixed = dup[(dup["count"] >= 5) & (dup["nunique"] > 1)]
affected_rows = mixed["count"].sum()
print(f"Profiles shared by >=5 pages with mixed real outcomes: {len(mixed)} profiles, "
      f"{affected_rows} rows ({affected_rows/len(df):.1%} of the dataset)")
print(mixed.sort_values("count", ascending=False).head(5))


Profiles shared by >=5 pages with mixed real outcomes: 919 profiles, 16858 rows (56.2% of the dataset)
                                             count  nunique      mean
content_age_days days_since_last_update ctr                          
482              22                     0.0    314        2  0.391720
313              104                    0.0    252        2  0.361111
463              22                     0.0    251        2  0.342629
117              20                     0.0    228        2  0.587719
441              22                     0.0    181        2  0.441989


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Re-score cadence:** monthly — matching the 30-day windows the underlying `trend_direction` label itself
is built from.

**Re-audit (don't just trust the next run) if:**
- A fresh grouped holdout's Precision@20 falls near or below 0.50 — the low end already observed in Week
  6's stability check. Below that, treat it as a real signal, not seed noise.
- Missingness rates for `word_count` / `search_volume` / `competition` drift far from today's baseline
  (25.7% / 8.2% / 8.2% — printed below); a new data pipeline could quietly change what's tracked.
- Archetype proportions shift sharply between runs (today: Watch 33%, Priority Refresh 24%, Protect 18%,
  Low Priority 15%, Quiet Decline 9%) without a known reason like a big new client.
- A new data export's `trend_direction` threshold doesn't match today's observed ~±20% cutoff — re-run the
  Week-6 check every refresh, since a silent definition change moves every downstream number.
- The client roster changes meaningfully (a large new client added) — Week 5–6 showed how much
  client-to-client heterogeneity affected results; a materially different portfolio mix should trigger
  re-validation, not just re-scoring.

In [10]:
print("Baseline reference values to diff future runs against:")
print(f"  Archetype proportions: {(df['archetype'].value_counts(normalize=True) * 100).round(1).to_dict()}")
for c in ["word_count", "search_volume", "competition"]:
    print(f"  {c} missingness: {df[c + '_missing'].mean():.1%}")

computed_trend = ((df["impressions_last_30d"] - df["impressions_prev_30d"])
                   / df["impressions_prev_30d"].replace(0, np.nan) * 100)
down_max = df.loc[df["trend_direction"] == "down", "trend_pct"].max()
print(f"  Observed 'down' threshold this run: trend_pct <= {down_max} (compare to ~-20 baseline)")


Baseline reference values to diff future runs against:
  Archetype proportions: {'Watch': 33.3, 'Priority Refresh': 23.9, 'Protect': 18.0, 'Low Priority': 15.3, 'Quiet Decline': 9.5}
  word_count missingness: 25.7%
  search_volume missingness: 8.2%
  competition missingness: 8.2%
  Observed 'down' threshold this run: trend_pct <= -20.0 (compare to ~-20 baseline)


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



Writing the full ranked queue to `work/outputs/` (regenerated every run, stays out of git per the CI
leak-guard), the metrics JSON to `work/outputs/` (committed — these are the receipts every number in this
notebook and the eventual paper traces back to), and two figures to `work/figures/` (committed, reused
directly in the paper): the archetype distribution, and the model-vs-baseline comparison from Week 6.

In [11]:
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --- Ranked queue CSV (regenerates every run; stays out of git) ---
output_cols = ["content_id", "client_id", "rank", "archetype", "reason_code", "action_label",
               "decline_proba", "expected_impact_score", "days_since_last_update",
               "impressions_90d", "avg_position", "ctr", "content_age_days"]
df[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(df)} rows to work/outputs/baseline_action_score.csv")

# --- Metrics JSON (committed — the receipts) ---
archetype_counts = df["archetype"].value_counts().reindex(
    ["Priority Refresh","Quiet Decline","Watch","Protect","Low Priority"])
archetype_decline_rates = df.groupby("archetype")["is_declining_label"].mean().reindex(archetype_counts.index)

metrics = {
    "generated_from": "w07_action_playbook.ipynb",
    "lane": "Lane 2 - Refresh / Content Opportunity Scoring",
    "label_definition": {
        "column": "trend_direction == 'down'",
        "paper_disclosed_threshold_pct": 10,
        "observed_local_threshold_pct": 20,
    },
    "validated_performance_w06": {
        "split": "GroupShuffleSplit by client_id, test_size=0.25",
        "base_rate": 0.517,
        "week4_baseline_rule": {"precision_at_20": 0.200, "precision_at_50": 0.300},
        "logistic_regression_single_split_seed42": {"precision_at_20": 0.750, "precision_at_50": 0.680},
        "logistic_regression_5_seed_stability": {
            "precision_at_20": {"mean": 0.680, "min": 0.500, "max": 0.750, "std": 0.093},
            "precision_at_50": {"mean": 0.636, "min": 0.480, "max": 0.800, "std": 0.109},
        },
        "random_forest_single_split_seed42": {"precision_at_20": 0.300, "precision_at_50": 0.440},
    },
    "leakage_checks": {
        "trend_pct_correlation_with_last30_vs_prev30": 0.9999999984,
        "with_vs_without_90d_aggregates_precision_at_20": {"with": 0.800, "without": 0.750},
    },
    "human_review_flag": {
        "profiles_with_5plus_pages_and_mixed_outcomes": int(len(mixed)),
        "rows_affected": int(affected_rows),
        "share_of_dataset": round(float(affected_rows / len(df)), 4),
    },
    "archetype_counts": archetype_counts.to_dict(),
    "archetype_decline_rates": {k: round(float(v), 4) for k, v in archetype_decline_rates.items()},
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w07_metrics.json")

# --- Figure 1: archetype distribution ---
fig, ax1 = plt.subplots(figsize=(8, 5))
colors = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60", "#95a5a6"]
ax1.bar(archetype_counts.index, archetype_counts.values, color=colors)
ax1.set_ylabel("Number of pages")
ax1.set_title("Content Action Playbook: pages per archetype")
plt.xticks(rotation=20, ha="right")
for i, (cnt, rate) in enumerate(zip(archetype_counts.values, archetype_decline_rates.values)):
    ax1.text(i, cnt + 150, f"{rate:.0%} declining", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("work/figures/archetype_distribution.png", dpi=150)
plt.close()
print("Wrote work/figures/archetype_distribution.png")

# --- Figure 2: model vs baseline (from Week 6) ---
methods = ["Base rate", "Week-4\nbaseline rule", "Logistic\nRegression"]
p20_vals = [0.517, 0.200, 0.750]
p50_vals = [0.517, 0.300, 0.680]
x = np.arange(len(methods))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, p20_vals, width, label="Precision@20", color="#2980b9")
ax.bar(x + width/2, p50_vals, width, label="Precision@50", color="#8e44ad")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Precision")
ax.set_title("Model vs. baseline (grouped-by-client holdout, seed=42)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("work/figures/model_vs_baseline.png", dpi=150)
plt.close()
print("Wrote work/figures/model_vs_baseline.png")

Wrote 30000 rows to work/outputs/baseline_action_score.csv
Wrote work/outputs/w07_metrics.json
Wrote work/figures/archetype_distribution.png
Wrote work/figures/model_vs_baseline.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.